# 训练后模型推理与可视化

本笔记本演示如何使用 `qwen3smvl/inference/inference_vl_model.py` 中定义的 `VLModelInference` 对已经训练好的模型检查点进行推理，并对生成结果进行可视化展示。

笔记本组织如下：

| Section | 内容 |
|---|---|
| 1 | 在 **DanQing** 数据集的测试分片上运行推理，并可视化图像、prompt、预测输出与参考答案 |
| 2 | 通过 `train.py` 的 `load_mixed_data_v2` 加载 **其他数据源** 的测试样本（例如 AlignMMBench），复用同一模型进行推理并可视化 |

**模型检查点**：`./model/unfreeze_connector_pretraining_test`


## 0. 环境与路径准备

In [ ]:
import sys
print('Python executable:', sys.executable)
print('Python version   :', sys.version.split()[0])

try:
    import transformers, accelerate
    print(f'transformers     : {transformers.__version__}')
    print(f'accelerate       : {accelerate.__version__}')
    from transformers import Trainer  # noqa: F401
    print('Trainer import   : OK')
except Exception as e:
    print('Trainer import   : FAIL →', type(e).__name__, e)
    print('\n→ Kernel is NOT using the project .venv, OR accelerate is missing there.')
    print('  Fix: in VSCode/Cursor, click the kernel picker (top-right of notebook)')
    print('       and select  .venv\\Scripts\\python.exe  under this project root.')

In [1]:
import os, sys, logging, random
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'CWD          = {os.getcwd()}')

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('inference_demo')
logging.getLogger('inference_vl_model').setLevel(logging.WARNING)

PROJECT_ROOT = E:\cursorprojects\Qwen3-SmVL
CWD          = E:\cursorprojects\Qwen3-SmVL


In [2]:
import torch
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
import textwrap

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    total_bytes = torch.cuda.get_device_properties(0).total_memory
    # 厂商标注的 "24 GB" 实际是二进制 GiB (2^30 bytes)；
    # 直接 / 1e9 得到 ~25.77 是 decimal GB，数值看起来偏大。
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {total_bytes / (1024 ** 3):.2f} GiB  '
          f'({total_bytes / 1e9:.2f} GB decimal)')

try:
    plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
except Exception as _e:
    print('字体设置警告:', _e)

e:\cursorprojects\Qwen3-SmVL\.venv\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Device: cuda:0
GPU: NVIDIA GeForce RTX 5090 Laptop GPU
VRAM: 23.89 GiB  (25.65 GB decimal)


### 可视化工具

封装一个 `show_sample` 工具，在一张 matplotlib figure 中同时展示：图像、用户 prompt、模型预测结果、参考答案（若有）。

In [3]:
def _to_pil(img):
    """归一化各种输入格式到 PIL.Image.Image。"""
    if isinstance(img, Image.Image):
        return img.convert('RGB')
    if isinstance(img, dict):
        if img.get('bytes'):
            return Image.open(BytesIO(img['bytes'])).convert('RGB')
        if img.get('path'):
            return Image.open(img['path']).convert('RGB')
    if isinstance(img, (bytes, bytearray)):
        return Image.open(BytesIO(img)).convert('RGB')
    if isinstance(img, str):
        return Image.open(img).convert('RGB')
    raise TypeError(f'无法识别的图像类型: {type(img)}')


def _wrap(text: str, width: int = 60) -> str:
    if text is None:
        return ''
    text = str(text).strip()
    return '\n'.join(textwrap.wrap(text, width=width, replace_whitespace=False, drop_whitespace=False) or [''])


def show_sample(image, prompt: str, prediction: str, reference: str = None,
                title: str = '', figsize=(12, 6)):
    """在同一张 figure 里并排展示图像和文本对比。"""
    pil_img = _to_pil(image)

    fig, (ax_img, ax_txt) = plt.subplots(1, 2, figsize=figsize, gridspec_kw={'width_ratios': [1, 1.3]})
    ax_img.imshow(pil_img)
    ax_img.set_axis_off()
    if title:
        ax_img.set_title(title, fontsize=11)

    ax_txt.set_axis_off()
    parts = [f'【Prompt】\n{_wrap(prompt)}', f'\n【Prediction】\n{_wrap(prediction)}']
    if reference:
        parts.append(f'\n【Reference】\n{_wrap(reference)}')
    ax_txt.text(0.0, 1.0, '\n'.join(parts), va='top', ha='left', fontsize=10, family='sans-serif', wrap=True)

    plt.tight_layout()
    plt.show()

## 1. DanQing 测试集推理与可视化

步骤：

1. 通过 `load_mixed_data_v2` 读取 `scripts/mixture/danqing.json` 定义的 DanQing 子集，取其中的 `test` 分片作为评测池；
2. 使用 `inference_vl_model.VLModelInference` 初始化模型，并加载训练后的权重；
3. 对选定数量的样本逐条调用 `generate`，收集预测；
4. 调用 `show_sample` 可视化每条样本。


In [6]:
from qwen3smvl.train.train import load_mixed_data_v2

DANQING_MIX = [
    {
        'source': 'danqing',
        'path':   'data/DanQing/data/train-*.parquet',
        'count':  -1,
        'label':  'DanQing',
    }
]

danqing_raw = load_mixed_data_v2(
    dataset_mix=DANQING_MIX,
    seed=42,
    test_size=3000,
)
danqing_test = danqing_raw['test']
print(f'DanQing test split: {len(danqing_test)} samples')
print('columns:', danqing_test.column_names)
print('first sample texts:', danqing_test[0]['texts'])

ImportError: cannot import name 'Trainer' from 'transformers' (e:\cursorprojects\Qwen3-SmVL\.venv\Lib\site-packages\transformers\__init__.py)

In [ ]:
from qwen3smvl.inference.inference_vl_model import VLModelInference

CHECKPOINT_PATH = os.path.join(PROJECT_ROOT, 'model', 'unfreeze_connector_pretraining_test')
print(f'CHECKPOINT_PATH = {CHECKPOINT_PATH}')
assert os.path.exists(os.path.join(CHECKPOINT_PATH, 'model.safetensors')), \
    f'未在 {CHECKPOINT_PATH} 找到 model.safetensors，请确认训练产物路径'

vl_infer = VLModelInference(
    checkpoint_path=CHECKPOINT_PATH,
    device=DEVICE,
    enable_csv=False,   # 笔记本中不需要 shape CSV 日志
)

In [ ]:
def sample_to_io(sample):
    """把 load_mixed_data_v2 样本拆成 (image, prompt, reference) 三元组。"""
    img = sample['images'][0] if sample['images'] else None
    turn = sample['texts'][0] if sample['texts'] else {}
    return img, turn.get('user', ''), turn.get('assistant', '')


NUM_DANQING = min(5, len(danqing_test))
danqing_indices = list(range(NUM_DANQING))
print(f'将在 {NUM_DANQING} 条 DanQing 测试样本上运行推理: {danqing_indices}')

danqing_results = []
for i in danqing_indices:
    sample = danqing_test[i]
    image, prompt, reference = sample_to_io(sample)
    if image is None:
        print(f'[WARN] 样本 #{i} 没有图像，跳过')
        continue

    prediction = vl_infer.generate(
        images=image,
        prompts=prompt,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        system_prompt='使用中文回答所有问题。',
    )
    danqing_results.append({
        'index':      i,
        'image':      image,
        'prompt':     prompt,
        'prediction': prediction,
        'reference':  reference,
    })
    print(f'✔  sample {i}: 生成 {len(str(prediction))} 字')

print(f'\n完成 DanQing 推理，共 {len(danqing_results)} 条结果。')

In [ ]:
for r in danqing_results:
    show_sample(
        image=r['image'],
        prompt=r['prompt'],
        prediction=r['prediction'],
        reference=r['reference'],
        title=f"DanQing sample #{r['index']}",
    )

## 2. 其他测试集推理与可视化（`load_mixed_data_v2` 加载）

这里演示用 `load_mixed_data_v2` 以 **flat-QA JSONL** 的形式载入 `AlignMMBench`（你可以把配置替换成任意 `cauldron / jsonl / json / parquet` 源）。AlignMMBench 的 `metadata.jsonl` 每行字段如下：

```
{"question_id": ..., "image_path": "images/...", "prompt": "...", "ref_answer": "...", ...}
```

因此我们通过 `user_field="prompt"`、`assistant_field="ref_answer"`、`image_field="image_path"` 把它映射到统一 schema。


In [ ]:
OTHER_MIX = [
    {
        'source':          'jsonl',
        'path':            'data/AlignMMBench/metadata.jsonl',
        'image_base_path': 'data/AlignMMBench',
        'image_field':     'image_path',
        'user_field':      'prompt',
        'assistant_field': 'ref_answer',
        'count':           64,
        'label':           'AlignMMBench',
    }
]

other_raw = load_mixed_data_v2(
    dataset_mix=OTHER_MIX,
    data_dir='data',
    seed=123,
    test_size=16,
)
other_test = other_raw['test']
print(f'Other test split: {len(other_test)} samples')
print('columns:', other_test.column_names)
print('first sample texts:', other_test[0]['texts'])

In [ ]:
NUM_OTHER = min(5, len(other_test))
rng = random.Random(2026)
other_indices = rng.sample(range(len(other_test)), NUM_OTHER)
print(f'从 other_test 中随机选取 {NUM_OTHER} 条样本: {other_indices}')

other_results = []
for i in other_indices:
    sample = other_test[i]
    image, prompt, reference = sample_to_io(sample)
    if image is None:
        print(f'[WARN] 样本 #{i} 没有图像，跳过')
        continue

    prediction = vl_infer.generate(
        images=image,
        prompts=prompt,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        system_prompt='使用中文回答所有问题。',
    )
    other_results.append({
        'index':      i,
        'image':      image,
        'prompt':     prompt,
        'prediction': prediction,
        'reference':  reference,
    })
    print(f'✔  sample {i}: 生成 {len(str(prediction))} 字')

print(f'\n完成 其他测试集 推理，共 {len(other_results)} 条结果。')

In [ ]:
for r in other_results:
    show_sample(
        image=r['image'],
        prompt=r['prompt'],
        prediction=r['prediction'],
        reference=r['reference'],
        title=f"Other sample #{r['index']}",
    )

---
### 扩展提示

- 想换成其他数据源（例如 Cauldron 子集、ShareGPT-4o、lnqa 等），只需替换 `OTHER_MIX` 中的 `source / path / field` 键即可，加载接口完全一致。
- `VLModelInference.generate` 同时支持批量（传入列表）与单样本（直接传 `PIL.Image` + `str`），可按显存酌情改为批量推理。
- 若需要把结果保存为 JSONL，可直接调用 `qwen3smvl.inference.inference_vl_model.save_results`。